# 01 · Exploring GSE316760 (primary cutaneous melanoma, 10x Visium)

Task due 26/10/26: *explore the expression matrices, spatial coordinates, metadata and tissue images using
Scanpy and Squidpy; check coordinate units and available annotations.*

GSE316760 is the simplest of our datasets: 2 samples (`mel2`, `mel3`) in the standard Space Ranger `.h5` format.
Background on all datasets: `docs/datasets.md`.

**Visium in one paragraph.** A tissue slice is placed on a slide with a hexagonal grid of ~5,000 spots
(55 µm wide, 100 µm apart). Each spot captures RNA from the cells above it (roughly 1–10 cells), so we get a
*genes × spots* count matrix plus the position of every spot on a microscope image of the same slice.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
from scipy.spatial import cKDTree

# Project root = the spatial-reasoning folder (the one with pyproject.toml), wherever the notebook runs from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STAGE = ROOT / "01_data_exploration"
sys.path.insert(0, str(ROOT / "src"))
from space_mel.io import load_visium

RAW = ROOT / "data/raw/GSE316760/files"
PROCESSED = ROOT / "data/processed"
FIGS = STAGE / "results/01_GSE316760"
FIGS.mkdir(parents=True, exist_ok=True)
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)

## 1. Load the samples

GEO renames and flattens Space Ranger's output folder, so `load_visium` first rebuilds the layout that
`squidpy.read.visium` expects (in `data/processed/visium/`, raw files untouched), then loads it.

In [ ]:
SAMPLES = {"mel2": "GSM9459774_mel2", "mel3": "GSM9459775_mel3"}

adatas = {
    lib: load_visium(RAW, prefix, PROCESSED / f"visium/GSE316760_{lib}", library_id=lib)
    for lib, prefix in SAMPLES.items()
}
adatas["mel2"]

## 2. What is inside an AnnData object

| Slot | Content here |
|---|---|
| `X` | the expression matrix: one row per **spot**, one column per **gene** (sparse, raw counts) |
| `obs` | per-spot table: `in_tissue`, `array_row`/`array_col` (grid position) |
| `var` | per-gene table: Ensembl ID, feature type, genome |
| `obsm["spatial"]` | spot centre (x, y) in **full-resolution image pixels** |
| `uns["spatial"][lib]` | tissue images (hires/lowres), scale factors, metadata |

In [ ]:
for lib, a in adatas.items():
    X = a.X
    is_int = np.allclose(X.data[:1000], np.round(X.data[:1000]))
    print(f"{lib}: {a.n_obs} spots x {a.n_vars} genes | {type(X).__name__}, "
          f"{X.nnz / (a.n_obs * a.n_vars):.1%} non-zero | integer counts: {is_int}")

a = adatas["mel2"]
display(a.obs.head())
display(a.var.head())

## 3. Metadata

Space Ranger stores the chemistry and software version in the `.h5` file. The gene panel tells us
which 10x protocol was used.

In [ ]:
for lib, a in adatas.items():
    meta = a.uns["spatial"][lib]["metadata"]
    n_mt = a.var_names.str.startswith("MT-").sum()
    print(f"{lib}: {meta} | in_tissue values: {a.obs.in_tissue.unique()} | MT- genes: {n_mt}")

**Reading this:**
- *Visium V1 slide*, Space Ranger 2.0.1, **17,943 genes**: this is the size of 10x's probe-based human
  transcriptome panel (used for FFPE tissue), not the full ~36k-gene transcriptome.
- The panel has **no mitochondrial (`MT-`) genes**, so the usual "% mitochondrial reads" quality check is not possible.
- All spots have `in_tissue == 1`: GEO provides the *filtered* matrix (spots under tissue only).

## 4. Tissue images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (lib, a) in zip(axes, adatas.items()):
    img = a.uns["spatial"][lib]["images"]["hires"]
    ax.imshow(img)
    ax.set_title(f"{lib}: hires image {img.shape[1]}x{img.shape[0]} px")
    ax.axis("off")
fig.savefig(FIGS / "tissue_images.png", bbox_inches="tight")

In [ ]:
pos = pd.read_csv(PROCESSED / "visium/GSE316760_mel2/spatial/tissue_positions.csv")
lib = adatas["mel2"].uns["spatial"]["mel2"]
scale = lib["scalefactors"]["tissue_hires_scalef"]   # full-res pixels -> hires image pixels

plt.figure(figsize=(7, 7))
plt.imshow(lib["images"]["hires"])
plt.scatter(pos.pxl_col_in_fullres * scale, pos.pxl_row_in_fullres * scale,
            c=pos.in_tissue.map({1: "black", 0: "red"}), s=3)
plt.title("black = kept (under tissue), red = removed (glass)")
plt.axis("off")

## 5. Coordinate units

`obsm["spatial"]` is in pixels of the original full-resolution microscope image (which GEO does not include:
we only get the downscaled *hires* and *lowres* versions). To plot spots on those images, multiply by
`tissue_hires_scalef` / `tissue_lowres_scalef`.

To get **micrometres** we need µm per pixel. Two independent ways:
1. Visium spots are 100 µm apart centre-to-centre, so `100 / (median nearest-neighbour distance in px)`.
2. `scalefactors["spot_diameter_fullres"]`: in Space Ranger this is the diameter of a theoretical
   **65 µm** spot (not the physical 55 µm), so `65 / spot_diameter_fullres`.

If both agree, we can trust the conversion.

In [ ]:
rows = []
for lib, a in adatas.items():
    sf = a.uns["spatial"][lib]["scalefactors"]
    xy = a.obsm["spatial"]
    nn_px = np.median(cKDTree(xy).query(xy, k=2)[0][:, 1])
    um_per_px_grid = 100 / nn_px
    um_per_px_sf = 65 / sf["spot_diameter_fullres"]
    a.obsm["spatial_um"] = xy * um_per_px_grid
    extent = np.ptp(a.obsm["spatial_um"], axis=0) / 1000
    rows.append({
        "sample": lib,
        "NN spot distance (px)": nn_px,
        "spot_diameter_fullres (px)": sf["spot_diameter_fullres"],
        "um/px from grid": um_per_px_grid,
        "um/px from scalefactor (65 um)": um_per_px_sf,
        "um/px if we wrongly used 55 um": 55 / sf["spot_diameter_fullres"],
        "tissue extent x * y (mm)": f"{extent[0]:.2f} x {extent[1]:.2f}",
    })
    a.uns["spatial_units"] = {"um_per_px": um_per_px_grid, "method": "100 um / median NN spot distance"}
pd.DataFrame(rows).set_index("sample").round(4)

**Conclusion:** both methods agree (≈0.346 µm per full-res pixel). Using the physical 55 µm spot size would
have under-estimated every distance by ~15%. We store micrometre coordinates in `obsm["spatial_um"]`;
all distance-based operations in this project should use them, because the pixel size differs between samples
and datasets.

## 6. Quality control: how much RNA does each spot capture?

`total_counts` = number of RNA molecules (UMIs) detected in a spot; `n_genes_by_counts` = how many different
genes. Low values often mark tissue edges, folds or damaged areas.

In [ ]:
for lib, a in adatas.items():
    sc.pp.calculate_qc_metrics(a, inplace=True, percent_top=None)
    print(lib, a.obs[["total_counts", "n_genes_by_counts"]].describe().loc[["min", "50%", "max"]].round(0).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for lib, a in adatas.items():
    axes[0].hist(a.obs.total_counts, bins=80, alpha=0.6, label=lib)
    axes[1].hist(a.obs.n_genes_by_counts, bins=80, alpha=0.6, label=lib)
axes[0].set_xlabel("total counts per spot"); axes[1].set_xlabel("genes detected per spot")
axes[0].legend()
fig.savefig(FIGS / "qc_histograms.png", bbox_inches="tight")

In [ ]:
for lib, a in adatas.items():
    sq.pl.spatial_scatter(a, library_id=lib, color=["total_counts", "n_genes_by_counts"],
                          img_alpha=0.5, figsize=(5, 5), save=FIGS / f"qc_spatial_{lib}.png")

## 7. Marker genes: where are tumour, immune and stroma?

GEO gives **no region annotations** (no "tumour" / "immune" labels), so as a first look we plot known markers.
We first normalise each spot to the same total and log-transform (standard scanpy workflow), keeping the raw
counts in `layers["counts"]`.

| Group | Genes |
|---|---|
| Melanoma / melanocytes | `MLANA`, `PMEL`, `TYR`, `SOX10` |
| Immune (all leukocytes) | `PTPRC` (CD45) |
| T cells | `CD3E`, `CD8A` |
| Macrophages / antigen presentation | `CD68`, `CD74` |
| Stroma (fibroblasts) | `COL1A1` |
| Epidermis (keratinocytes) | `KRT14` |

In [ ]:
MARKERS = ["MLANA", "PMEL", "SOX10", "PTPRC", "CD3E", "CD8A", "CD68", "CD74", "COL1A1", "KRT14"]

for lib, a in adatas.items():
    a.layers["counts"] = a.X.copy()
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    sq.pl.spatial_scatter(a, library_id=lib, color=MARKERS, img=False, ncols=5, figsize=(3.2, 3.2),
                          save=FIGS / f"markers_{lib}.png")

## 8. A first data-driven segmentation (clustering)

Since there are no annotations, we group spots with similar expression (Leiden clustering) and check where the
clusters sit and which genes define them. This is a *first look*, not a final annotation.

In [ ]:
for lib, a in adatas.items():
    sc.pp.highly_variable_genes(a, n_top_genes=2000, flavor="seurat_v3", layer="counts")
    sc.pp.pca(a, n_comps=30, mask_var="highly_variable")
    sc.pp.neighbors(a)
    sc.tl.leiden(a, resolution=0.5, flavor="igraph", n_iterations=2, key_added="leiden")
    sc.tl.umap(a)
    print(lib, a.obs.leiden.value_counts().sort_index().to_dict())

In [ ]:
for lib, a in adatas.items():
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    sc.pl.umap(a, color="leiden", ax=axes[0], show=False, title=f"{lib}: UMAP")
    sq.pl.spatial_scatter(a, library_id=lib, color="leiden", ax=axes[1], title=f"{lib}: clusters in space")
    fig.savefig(FIGS / f"clusters_{lib}.png", bbox_inches="tight")

In [ ]:
# Mean marker expression per cluster: which cluster looks like tumour, immune, stroma...?
for lib, a in adatas.items():
    sc.pl.matrixplot(a, MARKERS, groupby="leiden", standard_scale="var", title=lib,
                     cmap="viridis", show=False)
    plt.savefig(FIGS / f"cluster_markers_{lib}.png", bbox_inches="tight")
    plt.show()

In [ ]:
# Top genes distinguishing each cluster (compared with all other spots)
for lib, a in adatas.items():
    sc.tl.rank_genes_groups(a, "leiden", method="wilcoxon")
    top = pd.DataFrame(a.uns["rank_genes_groups"]["names"]).head(8)
    print(f"--- {lib}: top 8 genes per cluster")
    display(top)

## 9. Save

One `.h5ad` file per sample in `data/processed/` (gitignored). Later notebooks load these directly.

In [ ]:
for lib, a in adatas.items():
    path = PROCESSED / f"GSE316760_{lib}.h5ad"
    a.write_h5ad(path)
    print("saved", path.relative_to(ROOT), f"({path.stat().st_size / 1e6:.0f} MB)")

## 10. Summary and open questions

**What the data contains (task checklist)**
- **Expression matrices:** mel2 = 4,084 spots, mel3 = 4,949 spots, both × 17,943 genes. Raw integer UMI counts,
  sparse, already filtered to spots under tissue.
- **Metadata:** Visium V1 slide, Space Ranger 2.0.1, probe-based panel (FFPE). No mitochondrial genes, so no %MT QC.
  GEO gives no clinical metadata and no publication for this series.
- **Spatial coordinates:** `obsm["spatial"]` in full-resolution image pixels. **0.346 µm/px**, confirmed two ways
  (100 µm spot spacing and the 65 µm `spot_diameter_fullres`). Micrometre coordinates saved in `obsm["spatial_um"]`.
  The captured area is ~6.4 × 6.4 mm.
- **Tissue images:** H&E, *hires* (2000 px) and *lowres* (600 px) only. The full-resolution image is not on GEO.
- **Annotations:** none. Region labels (tumour / immune / stroma) have to be derived by us.

**First biological look** (my interpretation from marker genes; to be checked)
- **mel2:** melanocytic genes (PMEL, MLANA, TYRP1) are high across most of the section, so tumour fills most of
  it (clusters 0, 1). Epidermis along the right edge (KRT1/10/14, clusters 2, 8), hair follicles as diagonal streaks
  (KRT71/KRTAP, clusters 4, 6), and a macrophage-like patch (CD68, LAPTM5, TYROBP; cluster 3).
  T-cell genes (CD3E, CD8A) are almost absent.
- **mel3:** a clear tumour nodule in the lower half (clusters 0, 4) under dermis (COL1A1, DCN; cluster 2) with a
  plasma-cell/stroma region (IGKC, IGHG1; cluster 1). Antigen-presentation/immune genes (CD74, HLA-DRA, PTPRC;
  cluster 6) and an interferon/MHC-I programme (HLA-E, B2M, TAP1; cluster 7) sit around the tumour–stroma border.
  This is a good first test case for "immune-enriched regions near tumour".
- mel3 has two tumour clusters: 0 (MCAM, HMGA2, BCAN) vs 4 (PMEL, MLPH, APOE), possibly less vs more differentiated
  melanoma cell states (a theme of the GSE300445 paper). This is a hypothesis.

**Open questions / next steps**
1. Validate the cluster interpretations (the H&E and the literature).
2. Each spot mixes several cells. Consider deconvolution with a melanoma single-cell reference later.
3. Define tumour and immune regions robustly (e.g. `sc.tl.score_genes` with marker sets) as input for the first
   spatial operations: distance (µm) from immune spots to the nearest tumour spot, and neighbourhood enrichment
   (`sq.gr.nhood_enrichment`).

